# PySpark intorduction

## Row objects in PySpark

In [0]:
from pyspark.sql import Row

row1 = Row(name = "Bella", animal_type = "rabbit", age = 4)
row1

In [0]:
row2 = Row(name = "Doggu", animal_type = "dog", age = 10)
row2

## Create spark dataframe

can be created from
- Row
- list of dicts
- table
- csv
- parquet
- database
- existing dataframes ...

In [0]:
df = spark.createDataFrame([row1, row2])
df

In [0]:
df.show()

In [0]:
df.take(2)

In [0]:
display(df.take(2))

In [0]:
display(df)

In [0]:
employees = [
  {"name": "John D.", "age": 30, "department": "HR"},
  {"name": "Alice G.", "age": 25, "department": "Finance"},
  {"name": "Bob T.", "age": 35, "department": "IT"},
  {"name": "Eve A.", "age": 28, "department": "Marketing"}
]

df_employees = spark.createDataFrame(employees)
df_employees

In [0]:
display(df_employees)

In [0]:
df_employees.take(3)

## Read from csv file

In [0]:
from pathlib import Path

DATA_PATH = Path().resolve() / "data"

print(DATA_PATH)

df_athletes = spark.read.csv(str(DATA_PATH / "athlete_events.csv"), header = True)

display(df_athletes.take(5))

In [0]:
df_athletes.printSchema()

In [0]:
print(df_athletes.columns)

In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", FloatType()),
  StructField("Weight", FloatType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])

df_athletes_schema = spark.read.csv(str(DATA_PATH/"athlete_events.csv"), header=True, schema=schema)
display(df_athletes_schema.take(5))

## EDA

- calculate nulls

In [0]:
from pyspark.sql.functions import col, sum

nulls = df_athletes_schema.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_athletes_schema.columns])

display(nulls)

In [0]:
df_athletes_schema.groupBy("NOC").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN')"
).show()

### combining SQL and dataframes

In [0]:
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

df_swe_medals = spark.sql("""
    SELECT
        sport, count(medal) AS medals                  
    FROM df_athletes_schema
    WHERE noc = 'SWE' AND medal IN ('Gold', 'Silver', 'Bronze')
    GROUP BY sport
    ORDER BY medals DESC
""")

display(df_swe_medals)

In [0]:
fig = df_swe_medals.plot(kind="bar", x = "medals", y = "sport", title = "Swedish medals")

fig.update_layout(yaxis={"autorange": "reversed"})

## SQL cells

In [0]:
%sql
FROM df_athletes_schema LIMIT 1

In [0]:
%sql
SHOW CATALOGS;

CREATE CATALOG IF NOT EXISTS data;

CREATE SCHEMA IF NOT EXISTS data.olympics;



CREATE OR REPLACE TABLE data.olympics.sweden_medals AS
(
  SELECT
    name, age, weight, height, sport, medal
  FROM
    df_athletes_schema
  WHERE 
    noc = 'SWE' 
    AND medal IN ('Gold', 'Silver', 'Bronze')
);


FROM
  data.olympics.sweden_medals
LIMIT 4